In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

df = pd.read_csv(
    "Data/household_power_consumption.txt",
    sep=';',
    na_values='?'
)

power = df['Global_active_power'].dropna().values.astype(float)
power = power[10000:30000]   # 20,000 samples

print("Total raw samples:", len(power))

scaler = MinMaxScaler()
power = scaler.fit_transform(power.reshape(-1, 1)).flatten()

window_size = 16
X = []
y = []

threshold = np.percentile(power, 85)

for i in range(len(power) - window_size - 1):
    X.append(power[i:i+window_size])
    y.append(1 if power[i+window_size] > threshold else 0)

X = np.array(X)
y = np.array(y)

print("Total samples:", len(X))
print("Class distribution:", np.bincount(y))

split_index = int(0.8 * len(X))

X_train = X[:split_index]
X_test  = X[split_index:]

y_train = y[:split_index]
y_test  = y[split_index:]

print("Train size:", len(X_train))
print("Test size:", len(X_test))


model = MLPClassifier(
    hidden_layer_sizes=(4,2),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42
)

start = time.time()
model.fit(X_train, y_train)
print("Training time:", round(time.time() - start, 3), "sec")

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


y_probs = model.predict_proba(X_test)[:, 1]

thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]

for t in thresholds:
    
    y_pred_t = (y_probs > t).astype(int)
    
    cm = confusion_matrix(y_test, y_pred_t)
    report = classification_report(y_test, y_pred_t, output_dict=True)
    
    recall_peak = report['1']['recall']
    precision_peak = report['1']['precision']
    accuracy = accuracy_score(y_test, y_pred_t)
    
    print("\n-----------------------------------")
    print(f"Threshold: {t}")
    print("Accuracy:", round(accuracy, 4))
    print("Peak Recall:", round(recall_peak, 4))
    print("Peak Precision:", round(precision_peak, 4))
    print("Confusion Matrix:\n", cm)

X_0 = X_train[y_train == 0]
X_1 = X_train[y_train == 1]

y_0 = y_train[y_train == 0]
y_1 = y_train[y_train == 1]

n0 = len(y_0)
n1 = len(y_1)

print("Before balancing:")
print("Class 0:", n0)
print("Class 1:", n1)

# Proper oversampling to exact balance
if n1 < n0:
    indices = np.random.choice(n1, n0, replace=True)
    X_1_bal = X_1[indices]
    y_1_bal = y_1[indices]
else:
    X_1_bal = X_1
    y_1_bal = y_1

X_train_bal = np.vstack((X_0, X_1_bal))
y_train_bal = np.hstack((y_0, y_1_bal))

print("After balancing:")
print("Class distribution:", np.bincount(y_train_bal))

# Shuffle
shuffle_idx = np.random.permutation(len(X_train_bal))
X_train_bal = X_train_bal[shuffle_idx]
y_train_bal = y_train_bal[shuffle_idx]

# Train balanced model
model_bal = MLPClassifier(
    hidden_layer_sizes=(4,2),
    activation='tanh',
    solver='adam',
    max_iter=2000,
    random_state=42
)

start = time.time()
model_bal.fit(X_train_bal, y_train_bal)
print("Training time:", round(time.time() - start, 3), "sec")

y_pred_bal = model_bal.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_bal))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_bal))
print(classification_report(y_test, y_pred_bal))

print("Number of layers:", model_bal.n_layers_)
print("Layer sizes:", [coef.shape for coef in model_bal.coefs_])

for i, (w, b) in enumerate(zip(model_bal.coefs_, model_bal.intercepts_)):
    print(f"\nLayer {i} Weights shape:", w.shape)
    print(w)
    print(f"Layer {i} Bias shape:", b.shape)
    print(b)

Total raw samples: 20000
Total samples: 19983
Class distribution: [17000  2983]
Train size: 15986
Test size: 3997

===== MLP (Unbalanced Training) =====
Training time: 9.726 sec
Accuracy: 0.9707280460345259
Confusion Matrix:
 [[3550   60]
 [  57  330]]
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      3610
           1       0.85      0.85      0.85       387

    accuracy                           0.97      3997
   macro avg       0.92      0.92      0.92      3997
weighted avg       0.97      0.97      0.97      3997


===== Threshold Tuning (Unbalanced Model) =====

-----------------------------------
Threshold: 0.3
Accuracy: 0.9692
Peak Recall: 0.8992
Peak Precision: 0.8056
Confusion Matrix:
 [[3526   84]
 [  39  348]]

-----------------------------------
Threshold: 0.35
Accuracy: 0.97
Peak Recall: 0.8863
Peak Precision: 0.8186
Confusion Matrix:
 [[3534   76]
 [  44  343]]

-----------------------------------
Threshold: 0.4
Accu